# Batch vs Stochastic vs Mini-batch Gradient Descent

Three ways to use your data when computing each gradient update:

| Method | Samples per update | Updates per epoch | Character |
|--------|-------------------|-------------------|-----------|
| **Batch** | all N | 1 | smooth, slow, direct path |
| **Stochastic (SGD)** | 1 | N | noisy, fast, zig-zag path |
| **Mini-batch** | a small group (e.g. 10) | N / batch_size | balanced — the practical choice |

We'll fit a simple line `y = w·x + b` to noisy data and visualize how each method behaves.

## 1. Generate simple data

True relationship: `y = 2x + 1 + noise`. The goal is to recover `w = 2`, `b = 1`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
N = 100
x = np.random.uniform(-3, 3, N)
y = 2 * x + 1 + np.random.normal(0, 1, N)   # w_true=2, b_true=1

plt.scatter(x, y, s=12)
plt.xlabel('x'); plt.ylabel('y'); plt.title('Training data: y = 2x + 1 + noise')
plt.show()

## 2. One gradient-descent runner for all three

The **only** difference between the three methods is `batch_size`:

- `batch_size = N` -> **batch** (one update using all data per epoch)
- `batch_size = 1` -> **stochastic** (one update per sample)
- `batch_size = 10` -> **mini-batch**

For linear regression with MSE loss the gradients are analytic:

`dL/dw = mean(2·(pred - y)·x)`,  `dL/db = mean(2·(pred - y))`

We record the parameter trajectory `(w, b)`, the full-dataset loss after each epoch, and the per-update loss.

In [ ]:
def run_gd(batch_size, epochs=50, lr=0.01, start=(-2.0, 4.0), seed=0):
    w, b = start
    traj = [(w, b)]            # parameter path
    epoch_loss = []            # full-data loss once per epoch
    update_loss = []           # loss of each individual update (shows noise)
    idx = np.arange(N)
    rng = np.random.default_rng(seed)
    for ep in range(epochs):
        rng.shuffle(idx)       # shuffle order each epoch
        for s in range(0, N, batch_size):
            bi = idx[s:s + batch_size]
            xb, yb = x[bi], y[bi]
            pred = w * xb + b
            err = pred - yb
            grad_w = 2 * np.mean(err * xb)
            grad_b = 2 * np.mean(err)
            w -= lr * grad_w
            b -= lr * grad_b
            traj.append((w, b))
            update_loss.append(np.mean(err ** 2))
        full = w * x + b
        epoch_loss.append(np.mean((full - y) ** 2))
    return np.array(traj), np.array(epoch_loss), np.array(update_loss)

# run all three with identical lr, epochs, and starting point
traj_b,  eloss_b,  uloss_b  = run_gd(batch_size=N)    # batch
traj_s,  eloss_s,  uloss_s  = run_gd(batch_size=1)    # stochastic
traj_m,  eloss_m,  uloss_m  = run_gd(batch_size=10)   # mini-batch

print('final (w, b):')
print('  batch     ', traj_b[-1].round(3))
print('  stochastic', traj_s[-1].round(3))
print('  mini-batch', traj_m[-1].round(3))

## 3. Loss per epoch — convergence speed & stability

Full-dataset loss measured once at the end of each epoch. Watch how **batch** is smooth but slow to descend, **SGD** drops fast but stays jittery, and **mini-batch** sits in between.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(eloss_b, label='batch (size=N)', lw=2)
plt.plot(eloss_s, label='stochastic (size=1)', lw=2)
plt.plot(eloss_m, label='mini-batch (size=10)', lw=2)
plt.xlabel('epoch'); plt.ylabel('full-dataset MSE'); plt.yscale('log')
plt.title('Loss per epoch'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 4. Loss per update — the noise signature

Here we plot the loss of **each individual update** (the x-axis is rescaled to epochs so the three are comparable). This is where the difference is most visible:

- **Batch** — one smooth point per epoch, almost no noise.
- **SGD** — wildly noisy: each update only sees one sample, so its loss bounces around.
- **Mini-batch** — averaging over 10 samples smooths most of the noise out.

In [ ]:
plt.figure(figsize=(7, 4))
for uloss, lbl in [(uloss_b,'batch'), (uloss_s,'stochastic'), (uloss_m,'mini-batch')]:
    xs = np.linspace(0, 50, len(uloss))      # rescale update index -> epochs
    plt.plot(xs, uloss, label=lbl, alpha=0.7)
plt.xlabel('epoch'); plt.ylabel('per-update MSE'); plt.yscale('log')
plt.title('Loss at each parameter update'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 5. The path on the loss surface

The clearest picture: plot each method's trajectory of `(w, b)` over the MSE loss surface (contour lines). The true minimum is near `(2, 1)`.

- **Batch** takes a smooth, direct route to the minimum.
- **SGD** zig-zags noisily — each step is based on a single random sample.
- **Mini-batch** is a gentle middle ground.

In [ ]:
# build the loss surface over a grid of (w, b)
ws = np.linspace(-3, 5, 100)
bs = np.linspace(-2, 5, 100)
WW, BB = np.meshgrid(ws, bs)
Z = np.zeros_like(WW)
for i in range(WW.shape[0]):
    for j in range(WW.shape[1]):
        pred = WW[i, j] * x + BB[i, j]
        Z[i, j] = np.mean((pred - y) ** 2)

plt.figure(figsize=(7, 6))
plt.contour(WW, BB, Z, levels=30, cmap='viridis', alpha=0.6)
plt.plot(traj_s[:, 0], traj_s[:, 1], '-', color='tab:orange', alpha=0.5, label='stochastic')
plt.plot(traj_m[:, 0], traj_m[:, 1], '-', color='tab:green',  lw=1.5, label='mini-batch')
plt.plot(traj_b[:, 0], traj_b[:, 1], '-o', color='tab:blue', ms=3, label='batch')
plt.scatter([2], [1], color='red', marker='*', s=200, zorder=5, label='true (2, 1)')
plt.xlabel('w'); plt.ylabel('b'); plt.title('Optimization path on the loss surface')
plt.legend(); plt.show()

## Takeaways

- **Batch GD** — uses all data per step, so the gradient is exact: the path is smooth and heads straight for the minimum. But each step is expensive and it can be slow to make progress, especially on huge datasets.
- **Stochastic GD** — one sample per step: very fast, cheap updates, and the noise can even help escape bad spots — but it never settles cleanly, just bounces around the minimum.
- **Mini-batch GD** — the practical default everywhere in deep learning. It keeps most of SGD's speed and noise-robustness while being far smoother, and batches map perfectly onto GPU parallelism.

This is exactly why PyTorch's `DataLoader` uses a `batch_size`: you're choosing where on this spectrum to sit (typically 32-256).